# 03b — Upload Dataset to Hugging Face Hub

This notebook uploads the cleaned, split WDC-PAVE dataset (from notebook 03) to Hugging Face
as a public dataset repository.

**Prerequisites:**
- Notebook 03 has been run (train/val/test splits exist)
- `HF_TOKEN` is set in the project `.env` file
- `huggingface_hub` is installed (`uv pip install huggingface_hub`)

In [13]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import HfApi, login

load_dotenv(Path("../.env"))

HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

api = HfApi()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [14]:
# --- Configuration ---
# Change this to your HF username/org + desired repo name
REPO_ID = "siavashsaki/wdc-pave-ave"

CLEANED_DIR = Path("../data/WDC-PAVE/cleaned")
DATASET_CARD = Path("../data/WDC-PAVE/hf_dataset/README.md")

# Files to upload (path on disk -> path in the HF repo)
FILES = {
    CLEANED_DIR / "train.jsonl": "train.jsonl",
    CLEANED_DIR / "val.jsonl": "val.jsonl",
    CLEANED_DIR / "test.jsonl": "test.jsonl",
    CLEANED_DIR / "category_schemas.json": "category_schemas.json",
    CLEANED_DIR / "split_metadata.json": "split_metadata.json",
    DATASET_CARD: "README.md",
}

# Verify all files exist
for path in FILES:
    assert path.exists(), f"Missing: {path}"
    size_kb = path.stat().st_size / 1024
    print(f"  {path.name:30s} {size_kb:>8.1f} KB")
print(f"\nAll {len(FILES)} files ready for upload.")

  train.jsonl                       701.8 KB
  val.jsonl                         101.5 KB
  test.jsonl                        200.6 KB
  category_schemas.json               0.8 KB
  split_metadata.json                 0.8 KB
  README.md                           5.0 KB

All 6 files ready for upload.


In [15]:
# Create the dataset repo (no-op if it already exists)
repo_url = api.create_repo(
    repo_id=REPO_ID,
    repo_type="dataset",
    exist_ok=True,
)
print(f"Repository: {repo_url}")

Repository: https://huggingface.co/datasets/siavashsaki/wdc-pave-ave


In [16]:
# Upload all files in a single commit
from huggingface_hub import CommitOperationAdd

operations = [
    CommitOperationAdd(
        path_in_repo=path_in_repo,
        path_or_fileobj=str(local_path),
    )
    for local_path, path_in_repo in FILES.items()
]

commit_info = api.create_commit(
    repo_id=REPO_ID,
    repo_type="dataset",
    operations=operations,
    commit_message="Upload cleaned WDC-PAVE dataset (train/val/test splits)",
)
print(f"Commit: {commit_info.commit_url}")

No files have been modified since last commit. Skipping to prevent empty commit.


Commit: https://huggingface.co/datasets/siavashsaki/wdc-pave-ave/commit/2ba97f3d959e67a21953e64192fffcb63ce57512


In [17]:
# Verify: list files in the repo
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
print(f"Files in {REPO_ID}:")
for f in sorted(files):
    print(f"  {f}")

Files in siavashsaki/wdc-pave-ave:
  .gitattributes
  README.md
  category_schemas.json
  split_metadata.json
  test.jsonl
  train.jsonl
  val.jsonl


In [ ]:
# Verify: load the dataset back from HF to confirm it works
from datasets import load_dataset

ds = load_dataset(REPO_ID)
print(ds)
print(f"\nTrain sample:")
print(ds["train"][0])

## Done

The dataset is now available at `https://huggingface.co/datasets/{REPO_ID}`.

To load it in downstream notebooks:
```python
from datasets import load_dataset
ds = load_dataset("siavashsaki/wdc-pave-ave")
```